# Advanced Problems with Solutions: Metaclass Parameters

This notebook is a **problem-driven, runnable deep dive** into parameters passed in a class header:

```python
class MyClass(Base, metaclass=MyMeta, option=value):
    ...
```

It goes beyond the basic `__new__(..., **kwargs)` pattern and covers the full class-creation pipeline:

- metaclass selection and class-header keyword arguments
- `__prepare__`
- metaclass `__new__`
- metaclass `__init__`
- base-class `__init_subclass__`
- cooperative keyword consumption and forwarding
- namespace injection
- descriptors and `__set_name__`
- preserving `__classcell__`
- registries and uniqueness constraints
- metaclass composition
- inheritance-aware configuration
- dynamic class creation with `types.new_class`
- debugging unexpected keyword errors
- choosing between metaclasses, `__init_subclass__`, and class decorators

All examples are written to run top-to-bottom in a fresh Python 3 notebook.


## Mental Model

For a statement such as:

```python
class Service(Base, metaclass=ServiceMeta, region="eu", role="worker"):
    ...
```

the keywords after `metaclass=...` participate in class creation. Depending on your design:

1. `ServiceMeta.__prepare__` can inspect them before the class body executes.
2. `ServiceMeta.__new__` can consume metaclass-specific keywords.
3. Keywords forwarded to `type.__new__` can reach base classes through `__init_subclass__`.
4. `ServiceMeta.__init__` is also called with the original class-creation arguments, so a custom metaclass `__init__` should generally accept a compatible keyword surface.

A robust rule is:

> **Consume the keywords owned by the current layer, and cooperatively forward the rest.**


In [1]:
from __future__ import annotations

from collections import OrderedDict
from types import MappingProxyType, new_class
from typing import Any

print("Notebook initialized.")


Notebook initialized.


---
## Problem 1 — Accept and Validate Required Metaclass Parameters

Create a metaclass `VersionedMeta` that requires a named class-header parameter `api_version`.

Requirements:

- `api_version` must be a positive integer.
- The value must be stored on the resulting class as `__api_version__`.
- Invalid values should fail at **class creation time**, not instance creation time.
- Do not accidentally forward `api_version` to `type.__new__`.


### Solution

In [2]:
class VersionedMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        api_version,
        **kwargs,
    ):
        if not isinstance(api_version, int) or isinstance(api_version, bool):
            raise TypeError("api_version must be an integer")
        if api_version <= 0:
            raise ValueError("api_version must be > 0")

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__api_version__ = api_version
        return cls


class PaymentsAPI(metaclass=VersionedMeta, api_version=3):
    pass


assert PaymentsAPI.__api_version__ == 3
print(PaymentsAPI.__name__, PaymentsAPI.__api_version__)


PaymentsAPI 3


In [3]:
for bad_value in (0, -1, "3", True):
    try:
        class BadAPI(metaclass=VersionedMeta, api_version=bad_value):
            pass
    except (TypeError, ValueError) as exc:
        print(f"{bad_value!r:>4} -> {type(exc).__name__}: {exc}")


   0 -> ValueError: api_version must be > 0
  -1 -> ValueError: api_version must be > 0
 '3' -> TypeError: api_version must be an integer
True -> TypeError: api_version must be an integer


### Best-practice takeaway

Use explicit keyword-only parameters for required metaclass configuration:

```python
def __new__(..., *, required_option, **kwargs):
    ...
```

This makes the contract visible and causes a clear error when the option is omitted.


---
## Problem 2 — Understand `__prepare__`, `__new__`, and `__init__`

Build a metaclass that logs the parameters seen by each stage of class creation.

Question: do the custom class-header keywords reach only `__new__`, or can other metaclass hooks see them too?


### Solution

In [4]:
class LifecycleMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        print(f"__prepare__: name={name!r}, kwargs={kwargs}")
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        print(f"__new__:     name={name!r}, kwargs={kwargs}")
        # Consume all demo-only options instead of forwarding them to type.__new__.
        cls = super().__new__(mcls, name, bases, namespace)
        cls._creation_options = dict(kwargs)
        return cls

    def __init__(cls, name, bases, namespace, **kwargs):
        print(f"__init__:    name={name!r}, kwargs={kwargs}")
        # Same principle here: do not pass unsupported demo options to type.__init__.
        super().__init__(name, bases, namespace)


class LifecycleExample(
    metaclass=LifecycleMeta,
    trace=True,
    environment="test",
):
    answer = 42


assert LifecycleExample._creation_options == {
    "trace": True,
    "environment": "test",
}


__prepare__: name='LifecycleExample', kwargs={'trace': True, 'environment': 'test'}
__new__:     name='LifecycleExample', kwargs={'trace': True, 'environment': 'test'}
__init__:    name='LifecycleExample', kwargs={'trace': True, 'environment': 'test'}


Notice that the class-creation call has a shared keyword surface. If you override metaclass `__init__`, it should usually accept `**kwargs` even when the actual processing happens in `__new__`.


---
## Problem 3 — Split Parameters Between a Metaclass and `__init_subclass__`

Design this API:

```python
class Worker(
    BaseComponent,
    metaclass=AuditMeta,
    audit=True,          # owned by the metaclass
    role="background",   # owned by BaseComponent.__init_subclass__
):
    ...
```

Requirements:

- `AuditMeta` consumes `audit`.
- `role` must reach `BaseComponent.__init_subclass__`.
- Neither layer should steal parameters owned by another layer.


### Solution

In [5]:
class AuditMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        audit=False,
        **kwargs,
    ):
        # `audit` is ours, so consume it.
        # Forward the remaining keywords so base-class hooks can receive them.
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__audit_enabled__ = bool(audit)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        audit=False,
        **kwargs,
    ):
        # The same original keywords are supplied to metaclass __init__.
        # We accept them cooperatively.
        super().__init__(name, bases, namespace)


class BaseComponent(metaclass=AuditMeta):
    role = None

    def __init_subclass__(cls, *, role=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if role is not None and not isinstance(role, str):
            raise TypeError("role must be a string or None")
        cls.role = role


class BackgroundWorker(BaseComponent, audit=True, role="background"):
    pass


assert BackgroundWorker.__audit_enabled__ is True
assert BackgroundWorker.role == "background"
print(BackgroundWorker.__audit_enabled__, BackgroundWorker.role)


True background


### Why this pattern matters

`type.__new__` participates in the normal subclass-initialization protocol. Forwarding an unknown keyword can be correct when a base class owns it; forwarding a metaclass-owned keyword all the way down can instead produce an error.


---
## Problem 4 — Diagnose the Classic “Takes No Keyword Arguments” Failure

Explain and fix a class-creation failure caused by forwarding a keyword that nobody downstream consumes.


### Solution

In [6]:
class LeakyMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        # BAD for this option: forwards everything blindly.
        return super().__new__(mcls, name, bases, namespace, **kwargs)


try:
    class Broken(metaclass=LeakyMeta, debug=True):
        pass
except TypeError as exc:
    print("Expected failure:")
    print(" ", exc)


Expected failure:
  Broken.__init_subclass__() takes no keyword arguments


In [7]:
class FixedDebugMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        debug=False,
        **kwargs,
    ):
        # Consume the keyword owned by this metaclass.
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.debug = bool(debug)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        debug=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class Fixed(metaclass=FixedDebugMeta, debug=True):
    pass


assert Fixed.debug is True
print("Fixed.debug =", Fixed.debug)


Fixed.debug = True


### Debugging checklist

When class creation reports an unexpected keyword:

1. Identify which layer owns the keyword.
2. Check `metaclass.__prepare__`.
3. Check `metaclass.__new__`.
4. Check `metaclass.__init__`.
5. Check every relevant `__init_subclass__`.
6. Ensure each layer consumes its own parameters and forwards only the remainder.


---
## Problem 5 — Inject Attributes Before Class Creation

Create a metaclass parameter `constants={...}` that injects class attributes.

Requirements:

- Reject non-string names.
- Reject attempts to overwrite names already written in the class body.
- Inject the values **before** calling `type.__new__`.


### Solution

In [8]:
class ConstantInjectionMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        constants=None,
        **kwargs,
    ):
        constants = {} if constants is None else dict(constants)

        for attr_name, value in constants.items():
            if not isinstance(attr_name, str):
                raise TypeError("constant names must be strings")
            if attr_name in namespace:
                raise ValueError(
                    f"{name}: injected name {attr_name!r} "
                    "conflicts with the class body"
                )
            namespace[attr_name] = value

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__injected_constants__ = MappingProxyType(constants)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        constants=None,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class HttpStatus(
    metaclass=ConstantInjectionMeta,
    constants={"OK": 200, "NOT_FOUND": 404},
):
    description = "HTTP status subset"


assert HttpStatus.OK == 200
assert HttpStatus.NOT_FOUND == 404
assert dict(HttpStatus.__injected_constants__) == {
    "OK": 200,
    "NOT_FOUND": 404,
}
print(HttpStatus.OK, HttpStatus.NOT_FOUND)


200 404


In [9]:
try:
    class Conflict(
        metaclass=ConstantInjectionMeta,
        constants={"answer": 99},
    ):
        answer = 42
except ValueError as exc:
    print("Expected conflict:", exc)


Expected conflict: Conflict: injected name 'answer' conflicts with the class body


---
## Problem 6 — Why Injection Timing Matters for Descriptors

A descriptor's `__set_name__(owner, name)` is automatically called for descriptors present in the class namespace when `type.__new__` creates the class.

Compare:

- injecting a descriptor **before** `type.__new__`
- assigning it with `setattr` **after** the class already exists


### Solution

In [10]:
class NamedDescriptor:
    def __init__(self):
        self.owner = None
        self.name = None

    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name

    def __get__(self, instance, owner=None):
        return {
            "owner": None if self.owner is None else self.owner.__name__,
            "name": self.name,
        }


class DescriptorMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        inject_before=True,
        **kwargs,
    ):
        descriptor = NamedDescriptor()

        if inject_before:
            namespace["field"] = descriptor
            cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        else:
            cls = super().__new__(mcls, name, bases, namespace, **kwargs)
            setattr(cls, "field", descriptor)

        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        inject_before=True,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class Before(metaclass=DescriptorMeta, inject_before=True):
    pass


class After(metaclass=DescriptorMeta, inject_before=False):
    pass


print("Before.field ->", Before.field)
print("After.field  ->", After.field)

assert Before.field["name"] == "field"
assert After.field["name"] is None


Before.field -> {'owner': 'Before', 'name': 'field'}
After.field  -> {'owner': None, 'name': None}


If you intentionally attach a descriptor after class creation, call `descriptor.__set_name__(cls, name)` yourself when the descriptor protocol requires it.


In [11]:
late_descriptor = NamedDescriptor()
setattr(After, "late_field", late_descriptor)
late_descriptor.__set_name__(After, "late_field")

assert After.late_field == {"owner": "After", "name": "late_field"}
print(After.late_field)


{'owner': 'After', 'name': 'late_field'}


---
## Problem 7 — Use `__prepare__` to Detect Duplicate Definitions

Create a metaclass parameter `strict_names=True`.

When enabled, defining the same name twice in one class body should raise an error.

This demonstrates a case where the parameter must affect the namespace **before the class body executes**.


### Solution

In [12]:
class NoDuplicateNamespace(dict):
    def __setitem__(self, key, value):
        if key in self and key not in {"__module__", "__qualname__"}:
            raise NameError(f"duplicate class-body name: {key!r}")
        super().__setitem__(key, value)


class StrictNamespaceMeta(type):
    @classmethod
    def __prepare__(
        mcls,
        name,
        bases,
        *,
        strict_names=False,
        **kwargs,
    ):
        return NoDuplicateNamespace() if strict_names else {}

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        strict_names=False,
        **kwargs,
    ):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__strict_names__ = bool(strict_names)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        strict_names=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class StrictExample(metaclass=StrictNamespaceMeta, strict_names=True):
    x = 1
    y = 2


assert StrictExample.__strict_names__ is True
print(StrictExample.x, StrictExample.y)


1 2


In [13]:
try:
    class DuplicateExample(
        metaclass=StrictNamespaceMeta,
        strict_names=True,
    ):
        value = 1
        value = 2
except NameError as exc:
    print("Expected duplicate-definition failure:", exc)


Expected duplicate-definition failure: duplicate class-body name: 'value'


---
## Problem 8 — Preserve `__classcell__` When Rebuilding a Namespace

A metaclass may want to construct a fresh namespace dictionary. That is dangerous if it accidentally drops special compiler-generated entries such as `__classcell__`.

Create a safe metaclass that filters private names while preserving the information needed by methods that use zero-argument `super()` or `__class__`.


### Solution

In [14]:
class NamespaceFilteringMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        remove_private=False,
        **kwargs,
    ):
        if not remove_private:
            rebuilt = dict(namespace)
        else:
            rebuilt = {}
            for key, value in namespace.items():
                # Preserve essential interpreter/compiler keys.
                if key in {"__module__", "__qualname__", "__classcell__"}:
                    rebuilt[key] = value
                elif not key.startswith("_"):
                    rebuilt[key] = value

        return super().__new__(mcls, name, bases, rebuilt, **kwargs)

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        remove_private=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class ParentForClassCell:
    def label(self):
        return "parent"


class ChildForClassCell(
    ParentForClassCell,
    metaclass=NamespaceFilteringMeta,
    remove_private=True,
):
    _temporary = "will be filtered"

    def label(self):
        return super().label() + " -> child"


assert ChildForClassCell().label() == "parent -> child"
assert not hasattr(ChildForClassCell, "_temporary")
print(ChildForClassCell().label())


parent -> child


### Best practice

Prefer mutating the received namespace in place when reasonable. If you rebuild it, preserve interpreter-significant entries such as `__classcell__`.


---
## Problem 9 — Build a Validated Plugin Registry

Create a metaclass API:

```python
class JsonPlugin(
    Plugin,
    plugin_name="json",
    priority=20,
):
    ...
```

Requirements:

- `plugin_name` is required for concrete plugins.
- names must be unique
- `priority` must be an integer
- base/abstract classes can opt out with `abstract=True`
- provide an immutable registry view


### Solution

In [15]:
class PluginMeta(type):
    _registry = {}

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        plugin_name=None,
        priority=0,
        abstract=False,
        **kwargs,
    ):
        if not isinstance(priority, int) or isinstance(priority, bool):
            raise TypeError("priority must be an integer")

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        cls.plugin_name = plugin_name
        cls.priority = priority
        cls.abstract = bool(abstract)

        if not abstract:
            if not isinstance(plugin_name, str) or not plugin_name.strip():
                raise TypeError(
                    f"{name}: concrete plugins require a non-empty plugin_name"
                )

            if plugin_name in mcls._registry:
                previous = mcls._registry[plugin_name]
                raise ValueError(
                    f"duplicate plugin_name {plugin_name!r}: "
                    f"already used by {previous.__name__}"
                )

            mcls._registry[plugin_name] = cls

        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        plugin_name=None,
        priority=0,
        abstract=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)

    @classmethod
    def registry(mcls):
        return MappingProxyType(dict(mcls._registry))


class Plugin(metaclass=PluginMeta, abstract=True):
    pass


class JsonPlugin(Plugin, plugin_name="json", priority=20):
    pass


class CsvPlugin(Plugin, plugin_name="csv", priority=10):
    pass


registry = PluginMeta.registry()
assert registry["json"] is JsonPlugin
assert registry["csv"] is CsvPlugin
print({name: cls.priority for name, cls in registry.items()})


{'json': 20, 'csv': 10}


In [16]:
try:
    class DuplicateJsonPlugin(Plugin, plugin_name="json"):
        pass
except ValueError as exc:
    print("Expected registry collision:", exc)


Expected registry collision: duplicate plugin_name 'json': already used by JsonPlugin


---
## Problem 10 — Make Metaclass Configuration Inheritance-Aware

Sometimes a class parameter should be optional and inherit from a base class when omitted.

Design `ConfigMeta` with a `timeout=` parameter:

- If explicitly provided, validate it as a positive number.
- If omitted, inherit the nearest base's `timeout`.
- The root default is `30`.
- Distinguish “omitted” from “explicitly passed `None`”.


### Solution

In [17]:
_MISSING = object()


class ConfigMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        timeout=_MISSING,
        **kwargs,
    ):
        if timeout is _MISSING:
            inherited = next(
                (
                    base.timeout
                    for base in bases
                    if hasattr(base, "timeout")
                ),
                30,
            )
            resolved_timeout = inherited
        else:
            if (
                not isinstance(timeout, (int, float))
                or isinstance(timeout, bool)
                or timeout <= 0
            ):
                raise ValueError("timeout must be a positive number")
            resolved_timeout = timeout

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.timeout = resolved_timeout
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        timeout=_MISSING,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class RootService(metaclass=ConfigMeta):
    pass


class SlowService(RootService, timeout=120):
    pass


class SlowChild(SlowService):
    pass


assert RootService.timeout == 30
assert SlowService.timeout == 120
assert SlowChild.timeout == 120
print(RootService.timeout, SlowService.timeout, SlowChild.timeout)


30 120 120


In [18]:
try:
    class InvalidTimeout(RootService, timeout=None):
        pass
except ValueError as exc:
    print("Expected explicit-None failure:", exc)


Expected explicit-None failure: timeout must be a positive number


Using a unique sentinel is more precise than `None` when `None` could be a meaningful explicit value.


---
## Problem 11 — Compose Two Parameterized Metaclass Behaviors Cooperatively

Implement two metaclasses:

- `TagMeta` owns `tag=...`
- `FrozenMeta` owns `frozen=...`

Then combine them so this works:

```python
class Record(
    metaclass=CombinedMeta,
    tag="event",
    frozen=True,
):
    ...
```

Each metaclass should consume only its own parameter and call `super()`.


### Solution

In [19]:
class TagMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        tag=None,
        **kwargs,
    ):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.tag = tag
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        tag=None,
        **kwargs,
    ):
        super().__init__(name, bases, namespace, **kwargs)


class FrozenMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        frozen=False,
        **kwargs,
    ):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.frozen = bool(frozen)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        frozen=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace, **kwargs)


class CombinedMeta(TagMeta, FrozenMeta):
    pass


class EventRecord(
    metaclass=CombinedMeta,
    tag="event",
    frozen=True,
):
    pass


assert EventRecord.tag == "event"
assert EventRecord.frozen is True
print(EventRecord.tag, EventRecord.frozen)
print("CombinedMeta MRO:", [cls.__name__ for cls in CombinedMeta.__mro__])


event True
CombinedMeta MRO: ['CombinedMeta', 'TagMeta', 'FrozenMeta', 'type', 'object']


### Why cooperative `super()` matters

A hard-coded call such as `type.__new__(...)` inside `TagMeta` would skip `FrozenMeta.__new__` and break composition. Metaclass code benefits from the same cooperative multiple-inheritance discipline as ordinary classes.


---
## Problem 12 — Route Parameters to Multiple Base Classes

Suppose two base classes each define `__init_subclass__` parameters:

- `Cacheable` owns `cache=...`
- `Serializable` owns `format=...`

Make a subclass that supplies both options in one class header.


### Solution

In [20]:
class Cacheable:
    def __init_subclass__(cls, *, cache=False, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.cache_enabled = bool(cache)


class Serializable:
    def __init_subclass__(cls, *, format="json", **kwargs):
        super().__init_subclass__(**kwargs)
        if format not in {"json", "msgpack", "yaml"}:
            raise ValueError(f"unsupported format: {format!r}")
        cls.serialization_format = format


class CachedDocument(
    Cacheable,
    Serializable,
    cache=True,
    format="msgpack",
):
    pass


assert CachedDocument.cache_enabled is True
assert CachedDocument.serialization_format == "msgpack"
print(
    CachedDocument.cache_enabled,
    CachedDocument.serialization_format,
)


True msgpack


This does not require a custom metaclass at all. When your only goal is to configure subclasses, `__init_subclass__` is often the simpler tool.


---
## Problem 13 — Enforce Required Methods Based on a Metaclass Parameter

Build a metaclass with `requires=("method_a", "method_b")`.

Requirements:

- each required name must exist somewhere on the resulting class
- the attribute must be callable
- inherited implementations count
- errors should name all missing/non-callable requirements


### Solution

In [21]:
class RequirementMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        requires=(),
        **kwargs,
    ):
        requires = tuple(requires)
        if not all(isinstance(item, str) for item in requires):
            raise TypeError("requires must contain only strings")

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        problems = []
        for requirement in requires:
            if not hasattr(cls, requirement):
                problems.append(f"{requirement!r} is missing")
            elif not callable(getattr(cls, requirement)):
                problems.append(f"{requirement!r} is not callable")

        if problems:
            joined = "; ".join(problems)
            raise TypeError(f"{name} requirement failure: {joined}")

        cls.__requirements__ = requires
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        requires=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class HasSave:
    def save(self):
        return "saved"


class Repository(
    HasSave,
    metaclass=RequirementMeta,
    requires=("save", "load"),
):
    def load(self):
        return "loaded"


repo = Repository()
assert repo.save() == "saved"
assert repo.load() == "loaded"
print(repo.save(), repo.load())


saved loaded


In [22]:
try:
    class BrokenRepository(
        metaclass=RequirementMeta,
        requires=("save", "load"),
    ):
        save = 123
except TypeError as exc:
    print("Expected requirement failure:", exc)


Expected requirement failure: BrokenRepository requirement failure: 'save' is not callable; 'load' is missing


---
## Problem 14 — Generate Methods from Class-Header Parameters

Create a metaclass parameter:

```python
fields=("x", "y", "z")
```

Generate:

- `__slots__`
- an `__init__` accepting exactly those fields positionally
- a readable `__repr__`

Guard against:

- duplicate field names
- invalid identifiers
- fields beginning with `_`


### Solution

In [23]:
class RecordMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        fields=(),
        **kwargs,
    ):
        fields = tuple(fields)

        if len(set(fields)) != len(fields):
            raise ValueError("fields must be unique")

        for field in fields:
            if not isinstance(field, str) or not field.isidentifier():
                raise ValueError(f"invalid field name: {field!r}")
            if field.startswith("_"):
                raise ValueError(
                    f"private field names are not allowed: {field!r}"
                )

        if "__slots__" in namespace:
            raise ValueError("class body must not define __slots__ directly")

        namespace["__slots__"] = fields

        if "__init__" not in namespace:
            def __init__(self, *values):
                if len(values) != len(fields):
                    raise TypeError(
                        f"expected {len(fields)} values, got {len(values)}"
                    )
                for field, value in zip(fields, values):
                    setattr(self, field, value)

            namespace["__init__"] = __init__

        if "__repr__" not in namespace:
            def __repr__(self):
                payload = ", ".join(
                    f"{field}={getattr(self, field)!r}"
                    for field in fields
                )
                return f"{type(self).__name__}({payload})"

            namespace["__repr__"] = __repr__

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__fields__ = fields
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        fields=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class Point3D(metaclass=RecordMeta, fields=("x", "y", "z")):
    pass


p = Point3D(10, 20, 30)
assert (p.x, p.y, p.z) == (10, 20, 30)
print(p)


Point3D(x=10, y=20, z=30)


In [24]:
for fields in [
    ("x", "x"),
    ("valid", "not-valid"),
    ("_hidden",),
]:
    try:
        class BadRecord(metaclass=RecordMeta, fields=fields):
            pass
    except ValueError as exc:
        print(f"{fields!r} -> {exc}")


('x', 'x') -> fields must be unique
('valid', 'not-valid') -> invalid field name: 'not-valid'
('_hidden',) -> private field names are not allowed: '_hidden'


For production record-like classes, prefer `dataclasses`, `namedtuple`, `attrs`, or a validation library unless you specifically need metaclass-level behavior. This exercise is about the mechanics.


---
## Problem 15 — Dynamic Class Creation with the Same Parameterized Metaclass

Create a class dynamically with `types.new_class` while still passing custom class-creation parameters.

Requirements:

- use `ConfigMeta`
- pass `timeout=75`
- dynamically add a method `ping`


### Solution

In [25]:
def dynamic_body(namespace):
    namespace["kind"] = "dynamic"

    def ping(self):
        return f"pong after {type(self).timeout}s"

    namespace["ping"] = ping


DynamicService = new_class(
    "DynamicService",
    (),
    {
        "metaclass": ConfigMeta,
        "timeout": 75,
    },
    dynamic_body,
)


instance = DynamicService()
assert DynamicService.timeout == 75
assert instance.ping() == "pong after 75s"
print(DynamicService.timeout, instance.ping())


75 pong after 75s


---
## Problem 16 — Parameterized Class Ordering via `__prepare__`

Build a metaclass parameter `capture_order=True` that records the order in which user-defined class attributes appear.

Ignore compiler/interpreter keys such as `__module__` and `__qualname__`.


### Solution

In [26]:
class OrderedCaptureMeta(type):
    @classmethod
    def __prepare__(
        mcls,
        name,
        bases,
        *,
        capture_order=False,
        **kwargs,
    ):
        # Plain dicts preserve insertion order in modern Python.
        # OrderedDict is used here to make the intent explicit.
        return OrderedDict()

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        capture_order=False,
        **kwargs,
    ):
        # CPython may add additional internal dunder keys depending on
        # the Python version (for example __firstlineno__ or
        # __static_attributes__). Ignore compiler/interpreter dunders.
        ignored = {
            "__module__",
            "__qualname__",
            "__classcell__",
        }

        declared_names = tuple(
            key
            for key in namespace
            if key not in ignored
            and not (key.startswith("__") and key.endswith("__"))
        )

        cls = super().__new__(
            mcls,
            name,
            bases,
            dict(namespace),
            **kwargs,
        )

        cls.__declared_order__ = (
            declared_names if capture_order else ()
        )
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        capture_order=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class OrderedExample(
    metaclass=OrderedCaptureMeta,
    capture_order=True,
):
    first = 1

    def method(self):
        return "method"

    second = 2


print(OrderedExample.__declared_order__)
assert OrderedExample.__declared_order__ == (
    "first",
    "method",
    "second",
)


('first', 'method', 'second')


---
## Problem 17 — Validate Cross-Parameter Constraints

Create a metaclass for retry policies with:

- `retries` — non-negative integer
- `backoff` — non-negative number
- `jitter` — boolean

Constraint:

- `jitter=True` is valid only when `retries > 0` and `backoff > 0`.

Store the resolved policy as an immutable mapping.


### Solution

In [27]:
class RetryMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        retries=0,
        backoff=0.0,
        jitter=False,
        **kwargs,
    ):
        if (
            not isinstance(retries, int)
            or isinstance(retries, bool)
            or retries < 0
        ):
            raise ValueError("retries must be a non-negative integer")

        if (
            not isinstance(backoff, (int, float))
            or isinstance(backoff, bool)
            or backoff < 0
        ):
            raise ValueError("backoff must be a non-negative number")

        if not isinstance(jitter, bool):
            raise TypeError("jitter must be bool")

        if jitter and (retries == 0 or backoff == 0):
            raise ValueError(
                "jitter=True requires retries > 0 and backoff > 0"
            )

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.retry_policy = MappingProxyType(
            {
                "retries": retries,
                "backoff": float(backoff),
                "jitter": jitter,
            }
        )
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        retries=0,
        backoff=0.0,
        jitter=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class ResilientClient(
    metaclass=RetryMeta,
    retries=4,
    backoff=0.25,
    jitter=True,
):
    pass


print(dict(ResilientClient.retry_policy))
assert ResilientClient.retry_policy["retries"] == 4


{'retries': 4, 'backoff': 0.25, 'jitter': True}


In [28]:
try:
    class InvalidRetryClient(
        metaclass=RetryMeta,
        retries=0,
        backoff=0.25,
        jitter=True,
    ):
        pass
except ValueError as exc:
    print("Expected cross-parameter failure:", exc)


Expected cross-parameter failure: jitter=True requires retries > 0 and backoff > 0


---
## Problem 18 — Build a Small Production-Style Schema Metaclass

Create a more complete example combining several best practices.

API:

```python
class UserEvent(
    Event,
    schema_version=2,
    event_type="user.created",
    aliases=("user.new",),
):
    ...
```

Requirements:

- `Event` itself is abstract
- concrete events require `event_type`
- `schema_version` is a positive integer
- aliases are normalized to a tuple
- event types and aliases must be globally unique
- metadata is immutable
- class body methods remain normal Python methods


### Solution

In [29]:
class EventMeta(type):
    _types = {}

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        abstract=False,
        schema_version=1,
        event_type=None,
        aliases=(),
        **kwargs,
    ):
        if (
            not isinstance(schema_version, int)
            or isinstance(schema_version, bool)
            or schema_version <= 0
        ):
            raise ValueError("schema_version must be a positive integer")

        aliases = tuple(aliases)
        if not all(
            isinstance(alias, str) and alias
            for alias in aliases
        ):
            raise TypeError("aliases must contain non-empty strings")

        if event_type is not None and (
            not isinstance(event_type, str) or not event_type
        ):
            raise TypeError("event_type must be a non-empty string or None")

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        if not abstract:
            if event_type is None:
                raise TypeError(
                    f"{name}: concrete events require event_type"
                )

            names_to_register = (event_type, *aliases)

            if len(set(names_to_register)) != len(names_to_register):
                raise ValueError(
                    f"{name}: event_type/aliases contain duplicates"
                )

            collisions = [
                key
                for key in names_to_register
                if key in mcls._types
            ]
            if collisions:
                raise ValueError(
                    f"{name}: already-registered event names: "
                    f"{collisions}"
                )

            for key in names_to_register:
                mcls._types[key] = cls

        cls.__event_metadata__ = MappingProxyType(
            {
                "abstract": bool(abstract),
                "schema_version": schema_version,
                "event_type": event_type,
                "aliases": aliases,
            }
        )
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        abstract=False,
        schema_version=1,
        event_type=None,
        aliases=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)

    @classmethod
    def resolve(mcls, event_name):
        return mcls._types[event_name]


class Event(metaclass=EventMeta, abstract=True):
    def describe(self):
        metadata = self.__event_metadata__
        return (
            f"{metadata['event_type']} "
            f"v{metadata['schema_version']}"
        )


class UserCreated(
    Event,
    schema_version=2,
    event_type="user.created",
    aliases=("user.new",),
):
    pass


class UserDeleted(
    Event,
    schema_version=1,
    event_type="user.deleted",
):
    pass


assert EventMeta.resolve("user.created") is UserCreated
assert EventMeta.resolve("user.new") is UserCreated
assert EventMeta.resolve("user.deleted") is UserDeleted

print(UserCreated().describe())
print(dict(UserCreated.__event_metadata__))


user.created v2
{'abstract': False, 'schema_version': 2, 'event_type': 'user.created', 'aliases': ('user.new',)}


---
# Challenge Set

The following problems are intentionally less guided. A complete solution follows each prompt.


## Challenge 1 — Environment-Specific Method Injection

Create a metaclass parameter `environment` with allowed values `"dev"`, `"test"`, and `"prod"`.

- Always store the chosen environment.
- For non-production classes only, inject a `debug_info()` instance method.
- Reject unsupported environment names.
- Injection must happen before class creation.


### Challenge 1 Solution

In [30]:
class EnvironmentMeta(type):
    ALLOWED = {"dev", "test", "prod"}

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        environment="prod",
        **kwargs,
    ):
        if environment not in mcls.ALLOWED:
            raise ValueError(
                f"environment must be one of {sorted(mcls.ALLOWED)}"
            )

        if environment != "prod" and "debug_info" not in namespace:
            def debug_info(self):
                return {
                    "class": type(self).__name__,
                    "environment": type(self).environment,
                }

            namespace["debug_info"] = debug_info

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.environment = environment
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        environment="prod",
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class DevClient(metaclass=EnvironmentMeta, environment="dev"):
    pass


class ProdClient(metaclass=EnvironmentMeta, environment="prod"):
    pass


assert DevClient().debug_info()["environment"] == "dev"
assert not hasattr(ProdClient, "debug_info")
print(DevClient().debug_info())


{'class': 'DevClient', 'environment': 'dev'}


## Challenge 2 — Inheritance + Explicit Reset

Create a metaclass parameter `tags`.

Rules:

- if omitted, inherit tags from bases
- if supplied, append to inherited tags
- if `reset_tags=True`, ignore inherited tags
- remove duplicates while preserving order


### Challenge 2 Solution

In [31]:
class TagCollectionMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        tags=_MISSING,
        reset_tags=False,
        **kwargs,
    ):
        inherited = []
        if not reset_tags:
            for base in bases:
                inherited.extend(getattr(base, "tags", ()))

        local = [] if tags is _MISSING else list(tags)

        seen = set()
        resolved = []
        for tag in [*inherited, *local]:
            if not isinstance(tag, str):
                raise TypeError("all tags must be strings")
            if tag not in seen:
                seen.add(tag)
                resolved.append(tag)

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.tags = tuple(resolved)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        tags=_MISSING,
        reset_tags=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


class BaseTagged(metaclass=TagCollectionMeta, tags=("core", "shared")):
    pass


class ExtendedTagged(BaseTagged, tags=("shared", "api")):
    pass


class ResetTagged(
    ExtendedTagged,
    tags=("standalone",),
    reset_tags=True,
):
    pass


assert ExtendedTagged.tags == ("core", "shared", "api")
assert ResetTagged.tags == ("standalone",)
print(ExtendedTagged.tags)
print(ResetTagged.tags)


('core', 'shared', 'api')
('standalone',)


## Challenge 3 — A Parameterized Singleton Policy

Use a metaclass parameter `singleton=True/False`.

- Classes with `singleton=True` return the same instance every time.
- Classes with `singleton=False` behave normally.
- Keep the option per class, not globally.
- Implement instance behavior in metaclass `__call__`, not in class `__new__`.


### Challenge 3 Solution

In [32]:
class SingletonPolicyMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        singleton=False,
        **kwargs,
    ):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__singleton_enabled__ = bool(singleton)
        cls.__singleton_instance__ = None
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        singleton=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)

    def __call__(cls, *args, **kwargs):
        if not cls.__singleton_enabled__:
            return super().__call__(*args, **kwargs)

        if cls.__singleton_instance__ is None:
            cls.__singleton_instance__ = super().__call__(*args, **kwargs)

        return cls.__singleton_instance__


class SharedConnection(
    metaclass=SingletonPolicyMeta,
    singleton=True,
):
    def __init__(self, dsn="default"):
        self.dsn = dsn


class OrdinaryConnection(
    metaclass=SingletonPolicyMeta,
    singleton=False,
):
    pass


a = SharedConnection("db://first")
b = SharedConnection("db://second")
x = OrdinaryConnection()
y = OrdinaryConnection()

assert a is b
assert a.dsn == "db://first"
assert x is not y

print("shared:", a is b, a.dsn)
print("ordinary:", x is y)


shared: True db://first
ordinary: False


---
# Best Practices Summary

1. **Prefer keyword-only metaclass options.** They communicate intent and avoid ambiguity with base classes.
2. **Consume only what you own.** Remove a metaclass-specific option before forwarding remaining keywords.
3. **Use cooperative `super()`.** This is essential for composable metaclasses and multiple inheritance.
4. **Keep hook signatures compatible.** If you override metaclass `__init__`, accept the class-creation keyword surface too.
5. **Use `__prepare__` only for pre-body namespace behavior.** Otherwise, `__new__` is usually enough.
6. **Inject descriptors before `type.__new__`** when you want automatic `__set_name__`.
7. **Preserve `__classcell__`** when rebuilding class namespaces.
8. **Use a sentinel for “argument omitted”.** Do not overload `None` unless `None` truly means “not supplied”.
9. **Validate configuration during class creation.** Fail early and make errors name the class/parameter.
10. **Avoid mutable global registries without collision rules.** Expose read-only views where practical.
11. **Prefer `__init_subclass__` when a metaclass is unnecessary.** It is often simpler for subclass configuration.
12. **Prefer class decorators when transformation after class creation is enough.**
13. **Use metaclasses when behavior must participate in class construction itself**—for example, namespace preparation, metaclass-level registries, class factories, or instance-creation policy.
14. **Test failure paths.** Class-creation APIs are easiest to maintain when invalid declarations fail deterministically.


# Final Review Exercise

Without running the code first, predict what each keyword is owned by:

```python
class Job(
    FrameworkBase,
    metaclass=FrameworkMeta,
    trace=True,
    role="worker",
    timeout=60,
):
    ...
```

A clean architecture might assign:

- `trace` → `FrameworkMeta`
- `role` → `FrameworkBase.__init_subclass__`
- `timeout` → either the metaclass or a base hook, but **exactly one layer should own its semantics**

Then make the ownership explicit in signatures and forward only the unconsumed remainder.


In [33]:
print(
    "Completed:",
    "advanced metaclass-parameter problems and solutions."
)


Completed: advanced metaclass-parameter problems and solutions.
